# House Prices Regression - Modelling

## 1. Modelling Overview

This notebook focuses on preparing the data and training Machine Learning models to predict house sale prices.

The main steps include data cleaning, feature engineering, preprocessing, model training, model comparison, hyperparameter tuning and final evaluation.

The train-test split will be recreated using the same random state used in the EDA notebook to ensure consistency.

The test set will remain untouched until the final evaluation stage.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

## 2. Load Dataset

The original dataset is loaded from the raw data folder.

No transformations are applied at this stage.

In [2]:
df = pd.read_csv("../input/AmesHousing.csv")
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,Condition 2,Bldg Type,House Style,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Roof Style,Roof Matl,Exterior 1st,Exterior 2nd,Mas Vnr Type,Mas Vnr Area,Exter Qual,Exter Cond,Foundation,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin SF 1,BsmtFin Type 2,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Heating,Heating QC,Central Air,Electrical,1st Flr SF,2nd Flr SF,Low Qual Fin SF,Gr Liv Area,Bsmt Full Bath,Bsmt Half Bath,Full Bath,Half Bath,Bedroom AbvGr,Kitchen AbvGr,Kitchen Qual,TotRms AbvGrd,Functional,Fireplaces,Fireplace Qu,Garage Type,Garage Yr Blt,Garage Finish,Garage Cars,Garage Area,Garage Qual,Garage Cond,Paved Drive,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,5,1960,1960,Hip,CompShg,BrkFace,Plywood,Stone,112.0,TA,TA,CBlock,TA,Gd,Gd,BLQ,639.0,Unf,0.0,441.0,1080.0,GasA,Fa,Y,SBrkr,1656,0,0,1656,1.0,0.0,1,0,3,1,TA,7,Typ,2,Gd,Attchd,1960.0,Fin,2.0,528.0,TA,TA,P,210,62,0,0,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Feedr,Norm,1Fam,1Story,5,6,1961,1961,Gable,CompShg,VinylSd,VinylSd,NaN,0.0,TA,TA,CBlock,TA,TA,No,Rec,468.0,LwQ,144.0,270.0,882.0,GasA,TA,Y,SBrkr,896,0,0,896,0.0,0.0,1,0,2,1,TA,5,Typ,0,NaN,Attchd,1961.0,Unf,1.0,730.0,TA,TA,Y,140,0,0,0,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,6,1958,1958,Hip,CompShg,Wd Sdng,Wd Sdng,BrkFace,108.0,TA,TA,CBlock,TA,TA,No,ALQ,923.0,Unf,0.0,406.0,1329.0,GasA,TA,Y,SBrkr,1329,0,0,1329,0.0,0.0,1,1,3,1,Gd,6,Typ,0,NaN,Attchd,1958.0,Unf,1.0,312.0,TA,TA,Y,393,36,0,0,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,7,5,1968,1968,Hip,CompShg,BrkFace,BrkFace,NaN,0.0,Gd,TA,CBlock,TA,TA,No,ALQ,1065.0,Unf,0.0,1045.0,2110.0,GasA,Ex,Y,SBrkr,2110,0,0,2110,1.0,0.0,2,1,3,1,Ex,8,Typ,2,TA,Attchd,1968.0,Fin,2.0,522.0,TA,TA,Y,0,0,0,0,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,5,5,1997,1998,Gable,CompShg,VinylSd,VinylSd,NaN,0.0,TA,TA,PConc,Gd,TA,No,GLQ,791.0,Unf,0.0,137.0,928.0,GasA,Gd,Y,SBrkr,928,701,0,1629,0.0,0.0,2,1,3,1,TA,6,Typ,1,TA,Attchd,1997.0,Fin,2.0,482.0,TA,TA,Y,212,34,0,0,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [3]:
df.shape

(2930, 82)

## 3. Train-Test Split

The dataset is split into training and test sets before any preprocessing step.

The training set will be used for data cleaning, feature engineering, preprocessing and model training.

The test set will only be used at the end to evaluate the final selected model.

In [4]:
target = 'SalePrice'
X = df.drop(columns=[target])
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (2344, 81)
X_test shape: (586, 81)
y_train shape: (2344,)
y_test shape: (586,)


In [5]:
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()
y_train_clean = y_train.copy()
y_test_clean = y_test.copy()

print(f"X_train_clean shape: {X_train_clean.shape}")
print(f"X_test_clean shape: {X_test_clean.shape}")

X_train_clean shape: (2344, 81)
X_test_clean shape: (586, 81)


## 4. Initial Modelling Checklist

Before training models, the following steps must be performed:

1. Remove unnecessary identifier columns
2. Handle potential outliers in the training data
3. Treat missing values
4. Create engineered features
5. Encode categorical variables
6. Scale numerical variables when needed
7. Train a baseline model
8. Compare multiple regression models
9. Tune the best model
10. Evaluate the final model on the test set

## 5. Data Cleaning

In this section, we start cleaning the dataset before model training.

The first step is to remove identifier columns that do not provide predictive information for the model.

Columns such as `Order` and `PID` are useful for identifying records, but they should not be used as input features for machine learning models.

In [7]:
identifier_columns = ['Order', 'PID']
[col for col in identifier_columns if col in X_train_clean.columns]

['Order', 'PID']

In [8]:
X_train_clean = X_train_clean.drop(columns=identifier_columns)
X_test_clean = X_test_clean.drop(columns=identifier_columns)

print(f"X_train_clean shape: {X_train_clean.shape}")
print(f"X_test_clean shape: {X_test_clean.shape}")

X_train_clean shape: (2344, 79)
X_test_clean shape: (586, 79)


In [9]:
[col for col in identifier_columns if col in X_train_clean.columns]

[]

### 5.1 Outlier Treatment

Based on the EDA, some observations with extremely high `Gr Liv Area` may behave as outliers.

These observations can strongly influence regression models, especially linear models.

Outliers will be removed only from the training set.  
The test set will remain untouched to represent unseen real-world data.

In [10]:
train_clean_check = X_train_clean.copy()
train_clean_check["SalePrice"] = y_train_clean
gr_liv_area_outliers = train_clean_check[train_clean_check["Gr Liv Area"] > 4000]
gr_liv_area_outliers[
    [
        "Gr Liv Area",
        "SalePrice",
        "Overall Qual",
        "Neighborhood"
    ]
].sort_values(by="Gr Liv Area", ascending=False)

,Gr Liv Area,SalePrice,Overall Qual,Neighborhood
1498,5642,160000,10,Edwards
2180,5095,183850,10,Edwards
1760,4476,745000,10,NoRidge
1767,4316,755000,10,NoRidge


In [11]:
outlier_indexes = X_train_clean[X_train_clean["Gr Liv Area"] > 4000].index
print(f"Number of outliers to remove: {len(outlier_indexes)}")

X_train_clean = X_train_clean.drop(index=outlier_indexes)
y_train_clean = y_train_clean.drop(index=outlier_indexes)

print(f"X_train_clean shape after outlier removal: {X_train_clean.shape}")
print(f"y_train_clean shape after outlier removal: {y_train_clean.shape}")

Number of outliers to remove: 4
X_train_clean shape after outlier removal: (2340, 79)
y_train_clean shape after outlier removal: (2340,)


In [12]:
print(f"X_test_clean shape: {X_test_clean.shape}")
print(f"y_test_clean shape: {y_test_clean.shape}")

X_test_clean shape: (586, 79)
y_test_clean shape: (586,)


### Data Cleaning Notes

The identifier columns `Order` and `PID` were removed because they do not represent meaningful property characteristics for price prediction.

Extreme observations with `Gr Liv Area` greater than 4000 were removed from the training set only. These observations were identified during the EDA as potential outliers that could strongly affect regression models.

The test set was not modified during outlier removal. This is important because the test set must remain untouched until the final evaluation stage.

## 6. Feature Engineering

In this section, new features are created based on the insights from the exploratory data analysis.

The goal is to create variables that better represent property size, age, renovation status, bathroom capacity and the presence of important property characteristics.

The same transformations will be applied to both training and test sets to keep them consistent.